In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
from dotenv import load_dotenv
load_dotenv()
llm = model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [4]:
class JokesState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [5]:
def generate_joke(state: JokesState) -> JokesState:
    prompt = f"Generate a joke about {state['topic']}"
    response = llm.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokesState) -> JokesState:
    prompt = f"Explain why {state['topic']} is funny"
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [9]:
graph = StateGraph(JokesState)
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [13]:
config1 = {"configurable":{"thread_id":"1"}}
workflow.invoke({"topic":"usa"}, config=config1)

{'topic': 'usa',
 'joke': 'Why did the American flag go to therapy?\n\nBecause it was feeling a little "frayed" and had a lot of "star-spangled" issues to work through, but in the end, it just needed to "unite" itself and remember that it\'s still a "land of the free" to make jokes about itself.',
 'explanation': 'The United States of America (USA) - a country that can be quite amusing and entertaining to many people around the world. Here are some reasons why:\n\n1. **Cultural quirks**: American culture is a unique blend of different influences, which can lead to some humorous situations. For example, the obsession with large food portions, the love of reality TV shows, and the fascination with celebrity culture.\n2. **Politics and satire**: The USA has a long tradition of satire and political humor, with many comedians and writers using irony and sarcasm to comment on current events. Shows like "Saturday Night Live," "The Daily Show," and "South Park" are popular examples.\n3. **Regi

In [14]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'usa', 'joke': 'Why did the American flag go to therapy?\n\nBecause it was feeling a little "frayed" and had a lot of "star-spangled" issues to work through, but in the end, it just needed to "unite" itself and remember that it\'s still a "land of the free" to make jokes about itself.', 'explanation': 'The United States of America (USA) - a country that can be quite amusing and entertaining to many people around the world. Here are some reasons why:\n\n1. **Cultural quirks**: American culture is a unique blend of different influences, which can lead to some humorous situations. For example, the obsession with large food portions, the love of reality TV shows, and the fascination with celebrity culture.\n2. **Politics and satire**: The USA has a long tradition of satire and political humor, with many comedians and writers using irony and sarcasm to comment on current events. Shows like "Saturday Night Live," "The Daily Show," and "South Park" are popular e

In [ ]:
workflow.get_state_history(config=config1)

<generator object Pregel.get_state_history at 0x00000201389938A0>